# Phase 3 — Kaggle Inference (vLLM Few-Shot, Fine-Tuned Models)

Runs 3 fine-tuned SLMs with vLLM using few-shot prompting. Full fine-tuned weights (no LoRA).
Combines predictions via character-level majority voting.

| Model | Size | tp |
|-------|------|----||
| `qwen3_1_7b_finetuned` | 1.7B | 1 |
| `qwen3_8b_finetuned` | 8B | 2 |
| `lfm2_5_1_2b_finetuned` | 1.2B | 1 |

**Hardware**: T4 x2 on Kaggle (internet OFF)

| | |
|---|---|
| **Wheels** | `/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels` |
| **Qwen3-1.7B finetuned** | `/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-1-7b-finetuned/models/qwen3_1_7b_finetuned` |
| **Qwen3-8B finetuned** | `/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-8b-finetuned/models/qwen3_8b_finetuned` |
| **LFM2.5-1.2B finetuned** | `/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-lfm2-5-1-2b-finetuned/models/lfm2_5_1_2b_finetuned` |
| **Output** | `/kaggle/working/submission.csv` |

In [1]:
import os
import sys
import shutil
import subprocess
import site
from pathlib import Path

# =========================
# CONFIG
# =========================
WHEELS = "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels"
PKG_DIR = "/kaggle/working/pkgs"

# Same package specs — pip finds the matching wheels in WHEELS
PACKAGES = [
    "vllm==0.17.1",
    "transformers==4.56.0",
    "rapidfuzz>=3.0.0",
    "protobuf<6",
    "huggingface-hub>=0.34.0,<1.0",
    "msgspec>=0.18.0",
    "peft>=0.15.0",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.45.0",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--find-links", WHEELS,
    *PACKAGES,
])

# =========================
# VERIFY
# =========================
import torch
import transformers
import vllm
import rapidfuzz
import huggingface_hub
import msgspec
import google.protobuf

def where(mod):
    """Show where a module is loaded from."""
    return getattr(mod, "__file__", "builtin/namespace")

print("\n==== VERSION CHECK ====")
print(f"torch:            {torch.__version__}")
print(f"transformers:     {transformers.__version__}  ({where(transformers)})")
print(f"vllm:             {vllm.__version__}  ({where(vllm)})")
print(f"rapidfuzz:        {rapidfuzz.__version__}  ({where(rapidfuzz)})")
print(f"huggingface_hub:  {huggingface_hub.__version__}  ({where(huggingface_hub)})")
print(f"msgspec:          {msgspec.__version__}  ({where(msgspec)})")
print(f"protobuf:         {google.protobuf.__version__}  ({where(google.protobuf)})")

# Quick check: protobuf should now load from PKG_DIR
proto_path = where(google.protobuf)
if PKG_DIR not in proto_path:
    print(f"\n⚠ WARNING: protobuf still loading from outside PKG_DIR: {proto_path}")
else:
    print(f"\n✓ protobuf correctly loading from PKG_DIR")

print("\n==== CUDA ====")
print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory/1024**3:.1f} GB | CC {p.major}.{p.minor}")

Looking in links: /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/vllm-0.17.1-cp38-abi3-manylinux_2_31_x86_64.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/transformers-4.56.0-py3-none-any.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/rapidfuzz-3.14.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/huggingface_hub-0.36.2-py3-none-any.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/msgspec-0.21.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/bi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-adk 1.25.1 requires opentelemetry-api<1.40.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is incompatible.
google-adk 1.25.1 requires opentelemetry-sdk<1.40.0,>=1.36.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.



==== VERSION CHECK ====
torch:            2.10.0+cu128
transformers:     4.56.0  (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)
vllm:             0.17.1  (/usr/local/lib/python3.12/dist-packages/vllm/__init__.py)
rapidfuzz:        3.14.5  (/usr/local/lib/python3.12/dist-packages/rapidfuzz/__init__.py)
huggingface_hub:  0.36.2  (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)
msgspec:          0.21.1  (/usr/local/lib/python3.12/dist-packages/msgspec/__init__.py)
protobuf:         5.29.6  (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)

⚠ WARNING: protobuf still loading from outside PKG_DIR: /usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py

==== CUDA ====
CUDA available: True
GPU 0: Tesla T4 | 14.6 GB | CC 7.5
GPU 1: Tesla T4 | 14.6 GB | CC 7.5


In [2]:
import contextlib, gc, json, logging, re, sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from rapidfuzz.fuzz import partial_ratio_alignment
from tqdm import tqdm
from transformers import AutoTokenizer

from vllm import LLM, SamplingParams
from vllm.config import AttentionConfig
from vllm.v1.attention.backends.registry import AttentionBackendEnum
# GuidedDecodingParams removed in vLLM 0.12 → replaced by StructuredOutputsParams
from vllm.sampling_params import StructuredOutputsParams

try:
    from vllm.distributed.parallel_state import destroy_model_parallel
except ImportError:
    def destroy_model_parallel(): pass

print(f"vLLM version: {__import__('vllm').__version__}")

# T4 CC=7.5 — no native bfloat16
_DTYPE = torch.float16

CONFIG = {
    "DATA_DIR":              Path("/kaggle/input/competitions/nbme-score-clinical-patient-notes"),
    "OUTPUT_DIR":            Path("/kaggle/working"),
    "GPU_MEM_UTIL":          0.90,
    "MAX_MODEL_LEN":         1024,
    "MAX_NEW_TOKENS":        128,
    "LLM_TEMPERATURE":       0.0,
    "MAX_SPANS_PER_FEATURE": 10,
    "VOTE_THRESHOLD":        2,
    "FUZZY_SCORE_CUTOFF":    70.0,
    "SEED":                  42,
}

MODEL_REGISTRY = [
    {
        "name":              "lfm2_5_1_2b_finetuned",
        "model_path":        "/kaggle/input/datasets/natbrian/nbme-score-clinical-fine-tuned-models/brian/models/lfm2_5_1_2b_finetuned",
        "vllm_dtype":        "half",
        "tp":                1,
        "enable_thinking":   None,
        "trust_remote_code": True,
    },
    {
        "name":              "qwen3_1_7b_finetuned",
        "model_path":        "/kaggle/input/datasets/natbrian/nbme-score-clinical-fine-tuned-models/brian/models/qwen3_1_7b_finetuned",
        "vllm_dtype":        "half",
        "tp":                1,
        "enable_thinking":   False,
        "trust_remote_code": False,
    },
    {
        "name":              "qwen3_8b_finetuned",
        "model_path":        "/kaggle/input/datasets/natbrian/nbme-score-clinical-fine-tuned-models/brian/models/qwen3_8b_finetuned",
        "vllm_dtype":        "half",
        "tp":                2,
        "enable_thinking":   False,
        "trust_remote_code": False,
    },
]

# ── Few-shot examples ──────────────────────────────────────────────────────────
FEW_SHOT_EXAMPLES = [
    {
        "note":    "68 yo male with a 30-pack-year smoking history presents with hemoptysis and a 15-lb weight loss over 3 months.",
        "feature": "smoking history",
        "output":  '{"spans": ["30-pack-year smoking history"]}',
    },
    {
        "note":    "Patient is a 45 yo female with hypertension and type 2 diabetes mellitus. She takes metformin and lisinopril daily. She denies chest pain but reports occasional shortness of breath on exertion.",
        "feature": "current medications",
        "output":  '{"spans": ["metformin", "lisinopril"]}',
    },
    {
        "note":    "32 yo male presents with 3-day history of fever, productive cough with yellowish sputum, and left-sided pleuritic chest pain. Exam reveals decreased breath sounds at left base and dullness to percussion.",
        "feature": "pleuritic chest pain",
        "output":  '{"spans": ["left-sided pleuritic chest pain"]}',
    },
    {
        "note":    "55 yo woman with a history of rheumatoid arthritis managed with methotrexate. She presents for routine follow-up. No joint swelling noted today. Labs show WBC 3.2 and mild transaminase elevation.",
        "feature": "family history of autoimmune disease",
        "output":  '{"spans": []}',
    },
    {
        "note":    "Patient reports dull, aching pain in the right upper quadrant that worsens after fatty meals. She also notes nausea and one episode of vomiting. No jaundice.",
        "feature": "pain characteristics",
        "output":  '{"spans": ["dull, aching pain in the right upper quadrant", "worsens after fatty meals"]}',
    },
]

SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature.\n"
    "Rules:\n"
    "  1. Copy text character-for-character from the note — do NOT paraphrase or rephrase.\n"
    "  2. Only include spans that are literally present in the note.\n"
    "  3. If the feature is absent from the note, return an empty list.\n"
    "  4. Output ONLY valid JSON — no markdown, no explanation, no extra text.\n"
    'Format: {"spans": ["exact text 1", "exact text 2"]}'
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)
print("✓ CONFIG, MODEL_REGISTRY loaded")
print(f"  Models: {[m['name'] for m in MODEL_REGISTRY]}")
print(f"  Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")

2026-05-04 07:10:53.895720: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777878654.278039      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777878654.383609      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777878655.361263      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777878655.361312      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777878655.361315      22 computation_placer.cc:177] computation placer alr

vLLM version: 0.17.1
✓ CONFIG, MODEL_REGISTRY loaded
  Models: ['lfm2_5_1_2b_finetuned', 'qwen3_1_7b_finetuned', 'qwen3_8b_finetuned']
  Few-shot examples: 5


## Section 1 — Per-Note Regex FSM Constraint

In [3]:
def _build_char_class(note_chars: set) -> str:
    parts = []
    for ch in sorted(note_chars, key=ord):
        code = ord(ch)
        if code < 0x20 or code == 0x7F: continue
        if ch == ']':    parts.append(r'\]')
        elif ch == '^':  parts.append(r'\^')
        elif ch == '-':  parts.append(r'\-')
        elif ch == '\\': parts.append(r'\\')
        else:            parts.append(ch)
    return '[' + ''.join(parts) + ']' if parts else r'[^\n]'


def build_constraint_regex(pn_history: str, max_spans: int = 10) -> str:
    note_chars = set(pn_history) - {'"', '\\'}
    char_class = _build_char_class(note_chars)
    span_item  = f'"{char_class}*"'
    additional = r'(?:, ' + span_item + r'){0,' + str(max_spans - 1) + r'}'
    opt_list   = r'(?:' + span_item + additional + r')?'
    return r'\{"spans": \[' + opt_list + r'\]}'

print("✓ Section 1: build_constraint_regex defined")

✓ Section 1: build_constraint_regex defined


## Section 2 — Few-Shot Prompt Builder

In [4]:
def build_few_shot_prompt(feature_text: str, pn_history: str, tokenizer,
                          enable_thinking=None) -> str:
    """
    Build a chat prompt with few-shot examples followed by the target query.
    Alternates user/assistant turns so the model sees the expected output format.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Inject few-shot examples as alternating user/assistant turns
    for ex in FEW_SHOT_EXAMPLES:
        user_content = (
            f'Note: "{ex["note"]}"\n'
            f'Feature: {ex["feature"]}'
        )
        messages.append({"role": "user",      "content": user_content})
        messages.append({"role": "assistant", "content": ex["output"]})

    # Actual query
    query_content = (
        f'Note: "{pn_history.strip()}"\n'
        f'Feature: {feature_text}'
    )
    messages.append({"role": "user", "content": query_content})

    kwargs = dict(tokenize=False, add_generation_prompt=True)
    if enable_thinking is not None:
        try:
            return tokenizer.apply_chat_template(messages, enable_thinking=enable_thinking, **kwargs)
        except TypeError:
            pass
    return tokenizer.apply_chat_template(messages, **kwargs)

print("✓ Section 2: build_few_shot_prompt defined")

✓ Section 2: build_few_shot_prompt defined


## Section 3 — vLLM Engine Lifecycle

In [5]:
import shutil as _shutil
import tempfile as _tempfile

_STANDARD_TOKENIZER_CLASSES = {
    "PreTrainedTokenizerFast", "Qwen2Tokenizer", "Qwen2TokenizerFast",
    "LlamaTokenizer", "LlamaTokenizerFast", "GPT2Tokenizer", "GPT2TokenizerFast",
}

def _patch_model_config(model_path: str, **overrides) -> Path:
    """
    Symlink model dir into a writable temp dir and write patched config files.
    Patches applied:
      config.json:
        - use_cache=False → True  (vLLM platform detection reads from disk)
        - block_ff_dim injected from intermediate_size if missing  (LFM2 compat)
      tokenizer_config.json:
        - extra_special_tokens list → {}  (transformers 5.7.0 vs older format)
        - non-standard tokenizer_class → PreTrainedTokenizerFast  (offline, no auto_map)
    Returns path to temp dir; caller owns cleanup.
    """
    tmp = Path(_tempfile.mkdtemp(dir="/kaggle/working"))
    src = Path(model_path)
    PATCH_FILES = {"config.json", "tokenizer_config.json"}
    for item in src.iterdir():
        if item.name not in PATCH_FILES:
            (tmp / item.name).symlink_to(item.resolve())
    # Patch config.json
    with open(src / "config.json") as fh:
        cfg_dict = json.load(fh)
    # vLLM 0.17.1 lfm2.py reads config.block_ff_dim; transformers 5.7.0 saves intermediate_size instead
    if "block_ff_dim" not in cfg_dict and "intermediate_size" in cfg_dict:
        cfg_dict["block_ff_dim"] = cfg_dict["intermediate_size"]
        log.info(f"  Injecting block_ff_dim={cfg_dict['block_ff_dim']} from intermediate_size")
    cfg_dict.update(overrides)
    with open(tmp / "config.json", "w") as fh:
        json.dump(cfg_dict, fh, indent=2)
    # Patch tokenizer_config.json
    tok_cfg_path = src / "tokenizer_config.json"
    if tok_cfg_path.exists():
        with open(tok_cfg_path) as fh:
            tok_cfg = json.load(fh)
        # Fix extra_special_tokens list → dict (transformers 5.7.0 saves list, older expects dict)
        if isinstance(tok_cfg.get("extra_special_tokens"), list):
            tok_cfg["extra_special_tokens"] = {}
        # Fix non-standard tokenizer_class → PreTrainedTokenizerFast (offline, no auto_map)
        tok_cls = tok_cfg.get("tokenizer_class", "")
        if tok_cls and tok_cls not in _STANDARD_TOKENIZER_CLASSES:
            log.info(f"  Patching tokenizer_class {tok_cls!r} → PreTrainedTokenizerFast")
            tok_cfg["tokenizer_class"] = "PreTrainedTokenizerFast"
        with open(tmp / "tokenizer_config.json", "w") as fh:
            json.dump(tok_cfg, fh, indent=2)
    return tmp


def init_engine(model_path: str, model_spec: dict, cfg: dict) -> LLM:
    attn_cfg = AttentionConfig(backend=AttentionBackendEnum.TRITON_ATTN)
    patched = _patch_model_config(model_path, use_cache=True)
    try:
        log.info(f"  [{model_spec['name']}] Initialising vLLM engine (tp={model_spec['tp']}) ...")
        llm = LLM(
            model                  = str(patched),
            dtype                  = model_spec["vllm_dtype"],
            tensor_parallel_size   = model_spec["tp"],
            gpu_memory_utilization = cfg["GPU_MEM_UTIL"],
            max_model_len          = cfg["MAX_MODEL_LEN"],
            enforce_eager          = True,
            trust_remote_code      = model_spec.get("trust_remote_code", False),
            seed                   = cfg["SEED"],
            attention_config       = attn_cfg,
        )
    finally:
        _shutil.rmtree(patched, ignore_errors=True)
    log.info(f"  [{model_spec['name']}] vLLM engine ready.")
    return llm


def destroy_engine(llm, model_name: str) -> None:
    log.info(f"  [{model_name}] Destroying vLLM engine ...")
    destroy_model_parallel()
    with contextlib.suppress(Exception):
        torch.distributed.destroy_process_group()
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
        total_gb = torch.cuda.mem_get_info()[1] / 1024**3
        log.info(f"  [{model_name}] VRAM after cleanup: {free_gb:.1f}/{total_gb:.1f} GB free")

print("✓ Section 3: init_engine (config-patched), destroy_engine defined")

✓ Section 3: init_engine (config-patched), destroy_engine defined


## Section 4 — vLLM Inference Runner

In [6]:
def _parse_json_output(raw_text: str) -> list:
    raw_text = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(raw_text)
        return [s.strip() for s in parsed.get("spans", []) if isinstance(s, str) and s.strip()]
    except (json.JSONDecodeError, AttributeError):
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group())
                return [s.strip() for s in parsed.get("spans", []) if isinstance(s, str) and s.strip()]
            except json.JSONDecodeError:
                pass
    return []


def run_inference_vllm(llm, test_rows, pn_map, feat_map, tokenizer,
                       cfg, model_spec) -> list:
    model_name      = model_spec["name"]
    enable_thinking = model_spec.get("enable_thinking")
    log.info(f"  [{model_name}] Building few-shot prompts ...")
    prompts, params_list = [], []

    for _, row in test_rows.iterrows():
        pn_history   = pn_map.get(row["pn_num"], "").replace("\n", " ").strip()
        feature_text = feat_map.get((row["case_num"], row["feature_num"]), "")
        prompts.append(build_few_shot_prompt(feature_text, pn_history, tokenizer, enable_thinking))
        regex  = build_constraint_regex(pn_history, cfg["MAX_SPANS_PER_FEATURE"])
        # backend is set internally by vLLM 0.17.1 — do not pass to constructor
        params_list.append(SamplingParams(
            temperature=cfg["LLM_TEMPERATURE"],
            max_tokens=cfg["MAX_NEW_TOKENS"],
            structured_outputs=StructuredOutputsParams(regex=regex),
        ))

    log.info(f"  [{model_name}] Running vLLM inference on {len(prompts)} rows ...")
    outputs   = llm.generate(prompts=prompts, sampling_params=params_list)
    all_spans = []
    for output in tqdm(outputs, desc=f"  [{model_name}] Parsing", leave=False):
        raw = output.outputs[0].text.strip() if output.outputs else ""
        all_spans.append(_parse_json_output(raw))

    n_nonempty = sum(1 for s in all_spans if s)
    log.info(f"  [{model_name}] Done — non-empty: {n_nonempty}/{len(all_spans)}")
    return all_spans

print("✓ Section 4: _parse_json_output, run_inference_vllm defined")

✓ Section 4: _parse_json_output, run_inference_vllm defined


## Section 5 — Character-Level Majority Voting

In [7]:
def spans_to_char_array(span_locations: list, note_len: int) -> np.ndarray:
    arr = np.zeros(note_len, dtype=np.uint8)
    for start, end in span_locations:
        arr[max(0, start):min(note_len, end)] = 1
    return arr


def char_array_to_spans(arr: np.ndarray) -> list:
    spans, n, i = [], len(arr), 0
    while i < n:
        if arr[i] == 1:
            start = i
            while i < n and arr[i] == 1: i += 1
            spans.append((start, i))
        else:
            i += 1
    return spans


def locate_span_in_note(span_text: str, pn_history: str,
                        score_cutoff: float = 70.0) -> Optional[tuple]:
    span_text = span_text.strip()
    if not span_text or not pn_history: return None
    idx = pn_history.find(span_text)
    if idx != -1: return (idx, idx + len(span_text))
    idx = pn_history.lower().find(span_text.lower())
    if idx != -1: return (idx, idx + len(span_text))
    result = partial_ratio_alignment(span_text, pn_history, score_cutoff=score_cutoff)
    if result is not None: return (result.dest_start, result.dest_end)
    return None


def character_level_majority_vote(model_predictions, test_rows, pn_map,
                                  vote_threshold=2, fuzzy_cutoff=70.0) -> list:
    n_models, n_rows = len(model_predictions), len(test_rows)
    log.info(f"Majority vote ({n_models} models, threshold={vote_threshold}/{n_models}) ...")
    final_spans = []

    for seq_idx, (_, row) in enumerate(tqdm(test_rows.iterrows(), total=n_rows, desc="Majority vote")):
        pn_history = pn_map.get(row["pn_num"], "")
        note_len   = len(pn_history)
        if note_len == 0:
            final_spans.append([]); continue

        vote_array = np.zeros(note_len, dtype=np.int8)
        for preds in model_predictions:
            locs = [loc for text in preds[seq_idx]
                    if (loc := locate_span_in_note(text, pn_history, fuzzy_cutoff)) is not None]
            if locs:
                vote_array += spans_to_char_array(locs, note_len)

        consensus = (vote_array >= vote_threshold).astype(np.uint8)
        for i, ch in enumerate(pn_history):
            if ch in (' ', '\t', '\n', '\r') and consensus[i]:
                is_start = (i == 0 or consensus[i-1] == 0)
                is_end   = (i == note_len-1 or consensus[i+1] == 0)
                if is_start or is_end: consensus[i] = 0

        final_spans.append(char_array_to_spans(consensus))

    log.info(f"Vote complete — non-empty: {sum(1 for s in final_spans if s)}/{n_rows}")
    return final_spans

print("✓ Section 5: majority vote defined")

✓ Section 5: majority vote defined


## Section 6 — Submission Formatter

In [8]:
def format_location_string(spans: list, pn_history: str) -> str:
    if not spans: return ""
    clean = []
    for start, end in sorted(spans):
        while start < end and pn_history[start] in (' ', '\t', '\n', '\r'): start += 1
        while end > start and pn_history[end-1] in (' ', '\t', '\n', '\r'): end -= 1
        if start < end: clean.append((start, end))
    merged = []
    for start, end in sorted(clean):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return ";".join(f"{s} {e}" for s, e in merged) if merged else ""


def build_submission(final_spans: list, test_df: pd.DataFrame, pn_map: dict) -> pd.DataFrame:
    rows = []
    for row_idx, (_, test_row) in enumerate(test_df.iterrows()):
        pn_history = pn_map.get(test_row["pn_num"], "")
        spans      = final_spans[row_idx] if row_idx < len(final_spans) else []
        location   = format_location_string(spans, pn_history)
        rows.append({"id": test_row["id"], "location": location if location else np.nan})
    return pd.DataFrame(rows)

print("\u2713 Section 6: format_location_string, build_submission defined")

✓ Section 6: format_location_string, build_submission defined


## Run — Generate Submission

Pipeline per model:
1. Load fine-tuned model directly into vLLM (full fine-tuned weights, no LoRA)
2. Run batched inference with 5-shot examples + per-note regex constrained decoding
3. Destroy engine

Then: character-level majority vote → `submission.csv`

In [9]:
def main():
    cfg      = CONFIG
    data_dir = cfg["DATA_DIR"]

    print("\n" + "="*65)
    print("  PHASE 3: Kaggle Inference (vLLM Few-Shot, Fine-Tuned Models)")
    print(f"  Models: {[m['name'] for m in MODEL_REGISTRY]}")
    print(f"  Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")
    print("="*65 + "\n")

    print("▶ Loading test data ...")
    test_df  = pd.read_csv(data_dir / "test.csv")
    pn_df    = pd.read_csv(data_dir / "patient_notes.csv")
    feat_df  = pd.read_csv(data_dir / "features.csv")
    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = feat_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()
    print(f"  Test rows: {len(test_df)}")

    all_model_predictions = []

    for i, model_spec in enumerate(MODEL_REGISTRY):
        model_name = model_spec["name"]
        model_path = model_spec["model_path"]
        print(f"\n{'='*65}")
        print(f"  Model {i+1}/{len(MODEL_REGISTRY)}: {model_name}  (tp={model_spec['tp']})")
        print(f"  Path: {model_path}")
        print(f"{'='*65}")

        # Load tokenizer — use PreTrainedTokenizerFast directly when the model
        # has a custom tokenizer_class (e.g. TokenizersBackend for LFM2) that
        # isn't bundled as Python code in the model dir (no auto_map, offline Kaggle)
        trust_rc = model_spec.get("trust_remote_code", False)
        tok_cfg_path = Path(model_path) / "tokenizer_config.json"
        with open(tok_cfg_path) as _fh:
            _tok_cfg = json.load(_fh)
        _tok_cls = _tok_cfg.get("tokenizer_class", "")
        _standard_classes = {
            "PreTrainedTokenizerFast", "Qwen2Tokenizer", "Qwen2TokenizerFast",
            "LlamaTokenizer", "LlamaTokenizerFast", "GPT2Tokenizer", "GPT2TokenizerFast",
        }
        if _tok_cls and _tok_cls not in _standard_classes:
            # Custom tokenizer class without bundled Python code — load raw fast tokenizer
            from transformers import PreTrainedTokenizerFast
            log.info(f"  [{model_name}] tokenizer_class={_tok_cls!r} not importable; "
                     f"loading PreTrainedTokenizerFast from tokenizer.json")
            tokenizer = PreTrainedTokenizerFast.from_pretrained(
                model_path,
                extra_special_tokens={},
            )
        else:
            tokenizer = AutoTokenizer.from_pretrained(
                model_path,
                use_fast=True,
                trust_remote_code=trust_rc,
                extra_special_tokens={},  # tokenizer_config saved by transformers 5.7 uses list; older versions expect dict
            )
        if tokenizer.pad_token is None:
            tokenizer.pad_token    = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id

        llm = init_engine(model_path, model_spec, cfg)
        model_spans = run_inference_vllm(llm, test_df, pn_map, feat_map,
                                         tokenizer, cfg, model_spec)
        all_model_predictions.append(model_spans)

        destroy_engine(llm, model_name)
        del llm, tokenizer
        gc.collect()

    print("\n▶ Running character-level majority vote ...")
    effective_threshold = min(cfg["VOTE_THRESHOLD"], len(all_model_predictions))
    final_spans = character_level_majority_vote(
        all_model_predictions, test_df, pn_map,
        vote_threshold=effective_threshold,
        fuzzy_cutoff=cfg["FUZZY_SCORE_CUTOFF"],
    )

    submission_df = build_submission(final_spans, test_df, pn_map)
    out_path = cfg["OUTPUT_DIR"] / "submission.csv"
    submission_df.to_csv(out_path, index=False)

    print("\n" + "="*65)
    print(f"  ✓ Submission saved → {out_path}")
    print(f"  Shape: {submission_df.shape}")
    print(f"  Non-empty: {submission_df['location'].notna().sum()} / {len(submission_df)}")
    print("="*65)
    print(submission_df.head(10).to_string())

main()


  PHASE 3: Kaggle Inference (vLLM Few-Shot, Fine-Tuned Models)
  Models: ['lfm2_5_1_2b_finetuned', 'qwen3_1_7b_finetuned', 'qwen3_8b_finetuned']
  Few-shot examples: 5

▶ Loading test data ...


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'TokenizersBackend'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


  Test rows: 5

  Model 1/3: lfm2_5_1_2b_finetuned  (tp=1)
  Path: /kaggle/input/datasets/natbrian/nbme-score-clinical-fine-tuned-models/brian/models/lfm2_5_1_2b_finetuned
INFO 05-04 07:11:22 [utils.py:238] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'seed': 42, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=True, use_trtllm_attention=None, disable_flashinfer_prefill=False, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False), 'model': '/kaggle/working/tmpss21xh7i'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 05-04 07:11:46 [model.py:531] Resolved architecture: Lfm2ForCausalLM
WARNING 05-04 07:11:46 [model.py:1892] Casting torch.bfloat16 to torch.float16.
INFO 05-04 07:11:46 [model.py:1554] Using max model len 1024
INFO 05-04 07:11:47 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-04 07:11:47 [config.py:544] Setting attention block size to 16 tokens to ensure that attention page size is >= mamba page size.
INFO 05-04 07:11:47 [config.py:575] Padding mamba page size by 300.00% to ensure that mamba page size and attention page size are exactly equal.
INFO 05-04 07:11:47 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 05-04 07:11:47 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-04 07:11:47 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be i

...

(Worker pid=443) (Worker_TP1 pid=443) INFO 05-04 07:18:37 [multiproc_executor.py:749] Parent process exited, terminating worker
(Worker pid=442) (Worker_TP0 pid=442) INFO 05-04 07:18:37 [multiproc_executor.py:749] Parent process exited, terminating worker
(Worker pid=443) (Worker_TP1 pid=443) INFO 05-04 07:18:37 [multiproc_executor.py:802] WorkerProc shutting down.
(Worker pid=442) (Worker_TP0 pid=442) INFO 05-04 07:18:37 [multiproc_executor.py:802] WorkerProc shutting down.


nanobind: leaked 2 instances!
 - leaked instance 0x78789828a478 of type "xgrammar.xgrammar_bindings.CompiledGrammar"
 - leaked instance 0x78789828ab68 of type "xgrammar.xgrammar_bindings.GrammarMatcher"
nanobind: leaked 6 types!
 - leaked type "xgrammar.xgrammar_bindings.CompiledGrammar"
 - leaked type "xgrammar.xgrammar_bindings.TokenizerInfo"
 - leaked type "xgrammar.xgrammar_bindings.Grammar"
 - leaked type "xgrammar.xgrammar_bindings.GrammarMatcher"
 - leaked type "xgrammar.xgrammar_bindings.BatchGrammarMatcher"
 - leaked type "xgrammar.xgrammar_bindings.GrammarCompiler"
nanobind: leaked 51 functions!
 - leaked function "compile_builtin_json_grammar"
 - leaked function "__init__"
 - leaked function "from_vocab_and_metadata"
 - leaked function ""
 - leaked function "deserialize_json"
 - leaked function "compile_regex"
 - leaked function ""
 - leaked function "accept_string"
 - leaked function ""
 - leaked function "deserialize_json"
 - leaked function ""
 - leaked function "fill_nex


▶ Running character-level majority vote ...


Majority vote: 100%|██████████| 5/5 [00:00<00:00, 61.34it/s]


  ✓ Submission saved → /kaggle/working/submission.csv
  Shape: (5, 2)
  Non-empty: 5 / 5
          id location
0  00016_000  696 724
1  00016_001  668 676
2  00016_002  203 217
3  00016_003    70 91
4  00016_004  241 258
